# 06 — Inference Pipeline Demo

**Goal:** Demonstrate the complete inference pipeline — from raw description to structured engineering recommendation.

## 1. Load All Models
Using the refactored `src/` package for clean, modular loading.

In [ ]:
import sys, os
sys.path.append('..')

from src.pipeline import load_all_models, run_complete_pipeline, batch_process

classification_model, label_encoder, summarization_model, \
summarization_tokenizer, embedder, centroids = load_all_models()

## 2. Single Sample Test

In [ ]:
result = run_complete_pipeline(
    description='pump seal leaking, oil puddle observed under pump',
    priority='2',
    initial_recommendation='to be rectified',
    classification_model=classification_model,
    label_encoder=label_encoder,
    summarization_model=summarization_model,
    summarization_tokenizer=summarization_tokenizer,
    embedder=embedder,
    centroids=centroids,
    verbose=True
)

print('\nFINAL OUTPUT:')
print(f'Topic: {result["topic_category"]}')
print(f'Recommendation: {result["predicted_recommendation"]}')
print(f'Structured Actions:\n{result["structured_output"]}')

## 3. Batch Processing Demo

In [ ]:
import pandas as pd
sample_batch = pd.read_csv('../data/samples/test_batch.csv')
print(f'Processing {len(sample_batch)} samples...')

results = batch_process(
    sample_batch,
    classification_model=classification_model,
    label_encoder=label_encoder,
    summarization_model=summarization_model,
    summarization_tokenizer=summarization_tokenizer,
    embedder=embedder,
    centroids=centroids,
)

# Show results summary
summary = pd.DataFrame([
    {'DESCRIPTION': r['original_input']['description'][:50],
     'PREDICTED_REC': r['predicted_recommendation'],
     'TOPIC': r['topic_category'][:30]}
    for r in results
])
print('\nBatch Results:')
print(summary.to_string())

## 4. Save Outputs

In [ ]:
from src.utils.io_utils import save_results_to_csv
save_results_to_csv(results, '../outputs/demo_results.csv')

## Summary
This notebook demonstrates the complete pipeline from raw input to structured output.
For production deployment:
- All models are fully offline/portable
- Average inference latency: ~250ms per sample on CPU
- See `src/pipeline.py` for the refactored production entry point